<div style="
    text-align: center; 
    background: linear-gradient(135deg, #0062ff 0%, #00d4ff 100%); 
    font-family: 'Segoe UI', Roboto, Helvetica, Arial, sans-serif; 
    color: white; 
    padding: 35px 20px; 
    border-radius: 15px; 
    box-shadow: 0 10px 25px rgba(0, 98, 255, 0.3);
    margin-bottom: 25px;">
    <div style="font-size: 35px; font-weight: 800; letter-spacing: 1.5px; text-transform: uppercase; line-height: 1.2;">
        Trực Quan Hóa Dữ Liệu - Lab 03
    </div>
    <div style="font-size: 16px; font-weight: 500; margin-top: 10px; font-style: italic; opacity: 0.9;">
        "Xây dựng mô hình dữ liệu và trực quan hóa bằng Power BI"
    </div>
    <div style="font-size: 18px; font-weight: 600; margin-top: 15px; border-top: 1px solid rgba(255,255,255,0.4); display: inline-block; padding-top: 10px; letter-spacing: 1px;">
        NHÓM 05 - FIT-HCMUS
    </div>
</div>

<div style="text-align: center; font-size: 40px; font-weight: bold;">
  KHÁM PHÁ DỮ LIỆU PHIM TMDB (EDA)
</div>

## 1. Giới thiệu EDA và dữ liệu đầu vào

Mục tiêu của bước EDA này là thực hiện khảo sát, đánh giá sơ bộ cấu trúc và phân bố của tập dữ liệu sạch sau bước tiền xử lý trước khi tiến hành trực quan hóa trên Power BI.

**Phạm vi dữ liệu:**
- Tập trung nghiên cứu phim thuộc các quốc gia: Việt Nam, Hàn Quốc, Trung Quốc.
- Khoảng thời gian: Năm phát hành từ 2000 đến 2025.
- Các file dữ liệu đầu vào gồm file sạch tổng hợp và 7 file đã được tách bảng quan hệ Dim/Fact/Bridge.

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Cấu hình hiển thị dataframe và matplotlib
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)
sns.set_theme(style="whitegrid")

# Tự động điều chỉnh đường dẫn nếu chạy từ thư mục gốc hoặc thư mục 'notebooks'
if os.path.basename(os.getcwd()) == "notebooks":
    DATA_CLEAN_PATH = "../data/processed/tmdb_vn_kr_cn_2000_2025_scored.csv"
    POWERBI_DIR = "../data/processed/powerbi/"
else:
    DATA_CLEAN_PATH = "data/processed/tmdb_vn_kr_cn_2000_2025_scored.csv"
    POWERBI_DIR = "data/processed/powerbi/"

# Đường dẫn đến các bảng Dim/Fact/Bridge
DIM_PHIM_PATH = os.path.join(POWERBI_DIR, "DimPhim.csv")
FACT_HS_PATH = os.path.join(POWERBI_DIR, "FactHieuSuatPhim.csv")
DIM_TG_PATH = os.path.join(POWERBI_DIR, "DimThoiGian.csv")
DIM_QG_PATH = os.path.join(POWERBI_DIR, "DimQuocGia.csv")
BRIDGE_QG_PATH = os.path.join(POWERBI_DIR, "BridgePhimQuocGia.csv")
DIM_TL_PATH = os.path.join(POWERBI_DIR, "DimTheLoai.csv")
BRIDGE_TL_PATH = os.path.join(POWERBI_DIR, "BridgePhimTheLoai.csv")

print("Duong dan file du lieu sach tong hop:", DATA_CLEAN_PATH)
print("Duong dan thu muc Power BI:", POWERBI_DIR)

## 2. Đọc dữ liệu và kiểm tra cấu trúc các bảng

Đọc tất cả các bảng dữ liệu sau tiền xử lý, kiểm tra kích thước các bảng, hiển thị một số dòng dữ liệu mẫu và kiểm tra kiểu dữ liệu của các cột.

In [ ]:
# Doc cac bang du lieu
df_clean = pd.read_csv(DATA_CLEAN_PATH)
dim_phim = pd.read_csv(DIM_PHIM_PATH)
fact_hs = pd.read_csv(FACT_HS_PATH)
dim_tg = pd.read_csv(DIM_TG_PATH)
dim_qg = pd.read_csv(DIM_QG_PATH)
bridge_qg = pd.read_csv(BRIDGE_QG_PATH)
dim_tl = pd.read_csv(DIM_TL_PATH)
bridge_tl = pd.read_csv(BRIDGE_TL_PATH)

# In kich thuoc cac bang
print("Kich thuoc cac bang du lieu:")
print(f"  - df_clean:          {df_clean.shape}")
print(f"  - dim_phim:          {dim_phim.shape}")
print(f"  - fact_hs:           {fact_hs.shape}")
print(f"  - dim_tg:            {dim_tg.shape}")
print(f"  - dim_qg:            {dim_qg.shape}")
print(f"  - bridge_qg:         {bridge_qg.shape}")
print(f"  - dim_tl:            {dim_tl.shape}")
print(f"  - bridge_tl:         {bridge_tl.shape}")

# Hien thi thong tin bang Fact va Dim chinh
print("\nThong tin bang FactHieuSuatPhim:")
print(fact_hs.info())
display(fact_hs.head(5))

# Nguoi lam EDA viet tiep cac lenh kiem tra cau truc du lieu o day...

## 3. Tổng quan phạm vi dữ liệu

Thống kê tổng quan số lượng phim theo quốc gia sản xuất mục tiêu, theo năm phát hành và theo giai đoạn lịch sử để nhận định sự cân bằng về mặt số lượng giữa các phân khúc.

In [ ]:
# Thong ke so luong phim theo quoc gia chinh (primary)
print("Phan bo so luong phim theo quoc gia chinh:")
print(df_clean["QuocGiaChinh"].value_counts())
print("\nTy le phim theo quoc gia chinh:")
print(df_clean["QuocGiaChinh"].value_counts(normalize=True))

# Thong ke so phim theo nam
print("\nSo luong phim theo nam phat hanh:")
print(df_clean["NamPhatHanh"].value_counts().sort_index())

# Nguoi lam EDA viet tiep code thong ke va ve bieu do (vi du: line chart/bar chart) duoi day...

## 4. Khám phá thể loại và đặc điểm phim

Kiểm tra phân bố thể loại của phim, tìm ra top 10 thể loại phổ biến nhất và khảo sát xem có sự khác biệt rõ nét nào về thể loại đặc trưng (gu phim) giữa Việt Nam, Hàn Quốc và Trung Quốc hay không.

In [ ]:
# Ket noi cac bang de nghien cuu the loai phim
df_the_loai = bridge_tl.merge(dim_tl, on="MaTheLoai")
df_phim_tl = df_the_loai.merge(dim_phim, on="MaPhim")

# Thong ke top 10 the loai phim pho bien nhat
print("Top 10 the loai phim pho bien nhat:")
print(df_the_loai["TenTheLoai"].value_counts().head(10))

# Nguoi lam EDA viet tiep code phan tich the loai theo quoc gia va chuan bi ve bieu do o day...

## 5. Tổng quan các chỉ số đánh giá và chất lượng dữ liệu

Phân tích thống kê mô tả cho các cột chỉ số chính (Điểm đánh giá, Số lượt đánh giá, Độ phổ biến, Thời lượng, Doanh thu, Ngân sách) và phân bố của các cờ chất lượng dữ liệu.

In [ ]:
# Thong ke mo ta cac chi so dinh luong
cols_dinh_luong = ["DiemDanhGiaTB", "SoLuotDanhGia", "DoPhoBien", "ThoiLuong", "DoanhThu", "NganSach"]
print("Thong ke mo ta cho cac chi so dinh luong:")
display(fact_hs[cols_dinh_luong].describe())

# Phan bo theo cac co chat luong du lieu
flags = ["DuSoLuotDanhGia", "CoDoanhThu", "CoNganSach", "CoThoiLuong", "ThoiLuongBatThuong", "CanBoSungTenTiengAnh"]
print("\nPhan bo so luong phim theo cac co chat luong:")
for flag in flags:
    if flag in fact_hs.columns:
        print(f"\nPhan bo co {flag}:")
        print(fact_hs[flag].value_counts())
    elif flag in dim_phim.columns:
        print(f"\nPhan bo co {flag}:")
        print(dim_phim[flag].value_counts())

# Nguoi lam EDA viet tiep code ve bieu do boxplot/distribution plot hoac tinh correlation duoi day...

*Nguoi lam EDA viet nhan xet so bo ve phan bo va chat luong du lieu tai day...*

## Kết luận EDA sơ bộ

Phần này sẽ được bổ sung sau khi hoàn thành các thống kê và biểu đồ EDA.